In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

# Load clean data and movies
clean_data = pd.read_csv('../data/clean_data.csv')
movies = pd.read_csv('../data/ml-latest-small/movies.csv')

print("✅ Data loaded!")
print(f"Clean data shape: {clean_data.shape}")
print(f"Movies shape:     {movies.shape}")
print(movies.head())

✅ Data loaded!
Clean data shape: (100836, 5)
Movies shape:     (9742, 3)
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  


In [2]:
print("=== Sample genres ===")
print(movies['genres'].head(10))

# See all unique genres
all_genres = movies['genres'].str.split('|').explode().unique()
print(f"\nAll unique genres:")
print(sorted(all_genres))

=== Sample genres ===
0    Adventure|Animation|Children|Comedy|Fantasy
1                     Adventure|Children|Fantasy
2                                 Comedy|Romance
3                           Comedy|Drama|Romance
4                                         Comedy
5                          Action|Crime|Thriller
6                                 Comedy|Romance
7                             Adventure|Children
8                                         Action
9                      Action|Adventure|Thriller
Name: genres, dtype: str

All unique genres:
['(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']


In [3]:
# Replace pipe with space so TF-IDF reads each genre as a word
movies['genres_clean'] = movies['genres'].str.replace('|', ' ', regex=False)

# Remove (no genres listed)
movies['genres_clean'] = movies['genres_clean'].str.replace(
    '(no genres listed)', '', regex=False
).str.strip()

print("=== Cleaned genres ===")
print(movies[['title', 'genres_clean']].head(10))

=== Cleaned genres ===
                                title  \
0                    Toy Story (1995)   
1                      Jumanji (1995)   
2             Grumpier Old Men (1995)   
3            Waiting to Exhale (1995)   
4  Father of the Bride Part II (1995)   
5                         Heat (1995)   
6                      Sabrina (1995)   
7                 Tom and Huck (1995)   
8                 Sudden Death (1995)   
9                    GoldenEye (1995)   

                                  genres_clean  
0  Adventure Animation Children Comedy Fantasy  
1                   Adventure Children Fantasy  
2                               Comedy Romance  
3                         Comedy Drama Romance  
4                                       Comedy  
5                        Action Crime Thriller  
6                               Comedy Romance  
7                           Adventure Children  
8                                       Action  
9                    Action Adventu

In [4]:
# Extract year from title e.g. "Toy Story (1995)" → 1995
movies['year'] = movies['title'].str.extract(r'\((\d{4})\)')
movies['year'] = pd.to_numeric(movies['year'], errors='coerce')
movies['year'] = movies['year'].fillna(0).astype(int)

# Clean title (remove year)
movies['title_clean'] = movies['title'].str.replace(
    r'\s*\(\d{4}\)', '', regex=True
).str.strip()

print("=== Title and year extracted ===")
print(movies[['title', 'title_clean', 'year', 'genres_clean']].head(10))

=== Title and year extracted ===
                                title                  title_clean  year  \
0                    Toy Story (1995)                    Toy Story  1995   
1                      Jumanji (1995)                      Jumanji  1995   
2             Grumpier Old Men (1995)             Grumpier Old Men  1995   
3            Waiting to Exhale (1995)            Waiting to Exhale  1995   
4  Father of the Bride Part II (1995)  Father of the Bride Part II  1995   
5                         Heat (1995)                         Heat  1995   
6                      Sabrina (1995)                      Sabrina  1995   
7                 Tom and Huck (1995)                 Tom and Huck  1995   
8                 Sudden Death (1995)                 Sudden Death  1995   
9                    GoldenEye (1995)                    GoldenEye  1995   

                                  genres_clean  
0  Adventure Animation Children Comedy Fantasy  
1                   Adventure Ch

In [5]:
# Repeat genres twice to give them more weight than year
movies['combined_features'] = (
    movies['genres_clean'] + ' ' +
    movies['genres_clean'] + ' ' +
    movies['year'].astype(str)
)

movies['combined_features'] = movies['combined_features'].fillna('')

print("=== Combined features ===")
print(movies[['title', 'combined_features']].head(10))

=== Combined features ===
                                title  \
0                    Toy Story (1995)   
1                      Jumanji (1995)   
2             Grumpier Old Men (1995)   
3            Waiting to Exhale (1995)   
4  Father of the Bride Part II (1995)   
5                         Heat (1995)   
6                      Sabrina (1995)   
7                 Tom and Huck (1995)   
8                 Sudden Death (1995)   
9                    GoldenEye (1995)   

                                   combined_features  
0  Adventure Animation Children Comedy Fantasy Ad...  
1  Adventure Children Fantasy Adventure Children ...  
2                 Comedy Romance Comedy Romance 1995  
3     Comedy Drama Romance Comedy Drama Romance 1995  
4                                 Comedy Comedy 1995  
5   Action Crime Thriller Action Crime Thriller 1995  
6                 Comedy Romance Comedy Romance 1995  
7         Adventure Children Adventure Children 1995  
8                          

In [6]:
tfidf = TfidfVectorizer(
    min_df=2,
    max_features=5000,
    strip_accents='unicode',
    analyzer='word',
    token_pattern=r'\w{2,}',
    ngram_range=(1, 2),
    stop_words='english'
)

tfidf_matrix = tfidf.fit_transform(movies['combined_features'])

print(f"✅ TF-IDF matrix created!")
print(f"Shape: {tfidf_matrix.shape}")
print(f"Rows = movies, Columns = unique terms")
print(f"\nSample feature names:")
print(tfidf.get_feature_names_out()[:20])

✅ TF-IDF matrix created!
Shape: (9742, 1240)
Rows = movies, Columns = unique terms

Sample feature names:
['1916' '1920' '1923' '1924' '1925' '1926' '1927' '1928' '1929' '1930'
 '1931' '1932' '1933' '1934' '1935' '1936' '1937' '1938' '1939' '1940']


In [7]:
content_similarity = cosine_similarity(tfidf_matrix, tfidf_matrix)

content_similarity_df = pd.DataFrame(
    content_similarity,
    index=movies['title'],
    columns=movies['title']
)

print(f"✅ Content similarity matrix computed!")
print(f"Shape: {content_similarity_df.shape}")
print(f"\nTop similar movies to 'Toy Story (1995)':")
print(
    content_similarity_df['Toy Story (1995)']
    .sort_values(ascending=False)
    .head(10)
)

✅ Content similarity matrix computed!
Shape: (9742, 9742)

Top similar movies to 'Toy Story (1995)':
title
Toy Story (1995)                                           1.000000
Tale of Despereaux, The (2008)                             0.878527
Asterix and the Vikings (Astérix et les Vikings) (2006)    0.876737
Wild, The (2006)                                           0.876737
Monsters, Inc. (2001)                                      0.875263
The Good Dinosaur (2015)                                   0.874836
Toy Story 2 (1999)                                         0.874584
Shrek the Third (2007)                                     0.874232
Moana (2016)                                               0.873409
Adventures of Rocky and Bullwinkle, The (2000)             0.873310
Name: Toy Story (1995), dtype: float64


In [8]:
def content_recommend(movie_title, n_recommendations=10):
    """
    Given a movie title, return top N similar movies
    based on content (genres and year).
    """
    if movie_title not in content_similarity_df.columns:
        print(f"❌ '{movie_title}' not found.")
        print("Check the exact title and year e.g. 'Toy Story (1995)'")
        return None

    sim_scores = (
        content_similarity_df[movie_title]
        .sort_values(ascending=False)
        .drop(movie_title)
        .head(n_recommendations)
    )

    print(f"\n🎬 Movies similar to '{movie_title}' (content-based):")
    print("-" * 55)
    for i, (title, score) in enumerate(sim_scores.items(), 1):
        print(f"{i:2}. {title:<45} (score: {score:.4f})")

    return sim_scores

In [9]:
content_recommend('Toy Story (1995)')


🎬 Movies similar to 'Toy Story (1995)' (content-based):
-------------------------------------------------------
 1. Tale of Despereaux, The (2008)                (score: 0.8785)
 2. Asterix and the Vikings (Astérix et les Vikings) (2006) (score: 0.8767)
 3. Wild, The (2006)                              (score: 0.8767)
 4. Monsters, Inc. (2001)                         (score: 0.8753)
 5. The Good Dinosaur (2015)                      (score: 0.8748)
 6. Toy Story 2 (1999)                            (score: 0.8746)
 7. Shrek the Third (2007)                        (score: 0.8742)
 8. Moana (2016)                                  (score: 0.8734)
 9. Adventures of Rocky and Bullwinkle, The (2000) (score: 0.8733)
10. Emperor's New Groove, The (2000)              (score: 0.8733)


title
Tale of Despereaux, The (2008)                             0.878527
Asterix and the Vikings (Astérix et les Vikings) (2006)    0.876737
Wild, The (2006)                                           0.876737
Monsters, Inc. (2001)                                      0.875263
The Good Dinosaur (2015)                                   0.874836
Toy Story 2 (1999)                                         0.874584
Shrek the Third (2007)                                     0.874232
Moana (2016)                                               0.873409
Adventures of Rocky and Bullwinkle, The (2000)             0.873310
Emperor's New Groove, The (2000)                           0.873310
Name: Toy Story (1995), dtype: float64

In [10]:
content_recommend('Forrest Gump (1994)')


🎬 Movies similar to 'Forrest Gump (1994)' (content-based):
-------------------------------------------------------
 1. Colonel Chabert, Le (1994)                    (score: 0.8691)
 2. Tiger and the Snow, The (La tigre e la neve) (2005) (score: 0.8313)
 3. I Served the King of England (Obsluhoval jsem anglického krále) (2006) (score: 0.8297)
 4. Train of Life (Train de vie) (1998)           (score: 0.8261)
 5. Life Is Beautiful (La Vita è bella) (1997)    (score: 0.8246)
 6. Barry Lyndon (1975)                           (score: 0.7176)
 7. War and Peace (1956)                          (score: 0.7137)
 8. Edge of Love, The (2008)                      (score: 0.6876)
 9. Captain Corelli's Mandolin (2001)             (score: 0.6866)
10. Atonement (2007)                              (score: 0.6854)


title
Colonel Chabert, Le (1994)                                                0.869066
Tiger and the Snow, The (La tigre e la neve) (2005)                       0.831256
I Served the King of England (Obsluhoval jsem anglického krále) (2006)    0.829707
Train of Life (Train de vie) (1998)                                       0.826117
Life Is Beautiful (La Vita è bella) (1997)                                0.824637
Barry Lyndon (1975)                                                       0.717627
War and Peace (1956)                                                      0.713677
Edge of Love, The (2008)                                                  0.687555
Captain Corelli's Mandolin (2001)                                         0.686587
Atonement (2007)                                                          0.685429
Name: Forrest Gump (1994), dtype: float64

In [11]:
content_recommend('Pulp Fiction (1994)')


🎬 Movies similar to 'Pulp Fiction (1994)' (content-based):
-------------------------------------------------------
 1. Confessions of a Dangerous Mind (2002)        (score: 0.8412)
 2. Informant!, The (2009)                        (score: 0.8400)
 3. Leaves of Grass (2009)                        (score: 0.8400)
 4. Fargo (1996)                                  (score: 0.8380)
 5. Freeway (1996)                                (score: 0.8380)
 6. Beautiful Creatures (2000)                    (score: 0.8371)
 7. Party Monster (2003)                          (score: 0.8367)
 8. In Bruges (2008)                              (score: 0.8359)
 9. Man Bites Dog (C'est arrivé près de chez vous) (1992) (score: 0.8232)
10. Last Seduction, The (1994)                    (score: 0.7992)


title
Confessions of a Dangerous Mind (2002)                   0.841201
Informant!, The (2009)                                   0.840030
Leaves of Grass (2009)                                   0.840030
Fargo (1996)                                             0.837997
Freeway (1996)                                           0.837997
Beautiful Creatures (2000)                               0.837109
Party Monster (2003)                                     0.836724
In Bruges (2008)                                         0.835859
Man Bites Dog (C'est arrivé près de chez vous) (1992)    0.823154
Last Seduction, The (1994)                               0.799165
Name: Pulp Fiction (1994), dtype: float64

In [12]:
# Load collaborative model from Phase 3
with open('../models/collab_similarity.pkl', 'rb') as f:
    collab_similarity_df = pickle.load(f)

def compare_models(movie_title, n=5):
    print(f"\n{'='*60}")
    print(f"Movie: {movie_title}")
    print(f"{'='*60}")

    # Collaborative
    print(f"\n📊 Collaborative Filtering (user behaviour):")
    print("-" * 55)
    if movie_title in collab_similarity_df.columns:
        collab = (
            collab_similarity_df[movie_title]
            .sort_values(ascending=False)
            .drop(movie_title)
            .head(n)
        )
        for i, (title, score) in enumerate(collab.items(), 1):
            print(f"  {i}. {title:<45} ({score:.4f})")
    else:
        print("  Not found in collaborative model")

    # Content-based
    print(f"\n🎬 Content-Based (genres and year):")
    print("-" * 55)
    if movie_title in content_similarity_df.columns:
        content = (
            content_similarity_df[movie_title]
            .sort_values(ascending=False)
            .drop(movie_title)
            .head(n)
        )
        for i, (title, score) in enumerate(content.items(), 1):
            print(f"  {i}. {title:<45} ({score:.4f})")
    else:
        print("  Not found in content model")

compare_models('Toy Story (1995)')
compare_models('Forrest Gump (1994)')


Movie: Toy Story (1995)

📊 Collaborative Filtering (user behaviour):
-------------------------------------------------------
  1. Toy Story 2 (1999)                            (0.5726)
  2. Jurassic Park (1993)                          (0.5656)
  3. Independence Day (a.k.a. ID4) (1996)          (0.5643)
  4. Star Wars: Episode IV - A New Hope (1977)     (0.5574)
  5. Forrest Gump (1994)                           (0.5471)

🎬 Content-Based (genres and year):
-------------------------------------------------------
  1. Tale of Despereaux, The (2008)                (0.8785)
  2. Asterix and the Vikings (Astérix et les Vikings) (2006) (0.8767)
  3. Wild, The (2006)                              (0.8767)
  4. Monsters, Inc. (2001)                         (0.8753)
  5. The Good Dinosaur (2015)                      (0.8748)

Movie: Forrest Gump (1994)

📊 Collaborative Filtering (user behaviour):
-------------------------------------------------------
  1. Shawshank Redemption, The (1994)      

In [13]:
train_data, test_data = train_test_split(
    clean_data, test_size=0.2, random_state=42
)

def predict_rating_content(movie_title, content_sim_df, train_data, n=10):
    if movie_title not in content_sim_df.columns:
        return train_data['rating'].mean()

    similar = (
        content_sim_df[movie_title]
        .sort_values(ascending=False)
        .drop(movie_title)
        .head(n)
    )

    weighted_sum  = 0
    weight_total  = 0

    for title in similar.index:
        movie_ratings = train_data[train_data['title'] == title]['rating']
        if len(movie_ratings) > 0:
            avg_rating    = movie_ratings.mean()
            weight        = similar[title]
            weighted_sum  += weight * avg_rating
            weight_total  += weight

    if weight_total == 0:
        return train_data['rating'].mean()

    return weighted_sum / weight_total

# Evaluate on 500 samples
test_sample = test_data.sample(500, random_state=42)
actuals     = []
predictions = []

for _, row in test_sample.iterrows():
    predicted = predict_rating_content(
        row['title'],
        content_similarity_df,
        train_data
    )
    actuals.append(row['rating'])
    predictions.append(predicted)

rmse_content = np.sqrt(mean_squared_error(actuals, predictions))
print(f"✅ Content-Based RMSE: {rmse_content:.4f}")
print(f"\n  Below 1.0 = Good ✅")
print(f"  Your model: {'Good ✅' if rmse_content < 1.0 else 'Needs improvement ⚠️'}")
print(f"\nThe hybrid model in Phase 5 will combine both to beat these scores!")

✅ Content-Based RMSE: 1.1351

  Below 1.0 = Good ✅
  Your model: Needs improvement ⚠️

The hybrid model in Phase 5 will combine both to beat these scores!


In [14]:
import os
os.makedirs('../models', exist_ok=True)

# Save content similarity matrix
with open('../models/content_similarity.pkl', 'wb') as f:
    pickle.dump(content_similarity_df, f)

# Save movies with features
movies.to_csv('../data/movies_with_features.csv', index=False)

print("✅ Content-based model saved!")
print("   - models/content_similarity.pkl")
print("   - data/movies_with_features.csv")

✅ Content-based model saved!
   - models/content_similarity.pkl
   - data/movies_with_features.csv
